# Tutorial: Section 3.1 — Discrete Variables

This notebook is a guided tutorial for Section 3.1 of *Deep Learning: Foundations and Concepts* by Christopher M. Bishop and Hugh Bishop.

Section 3.1 introduces three closely related distributions:

1. **Bernoulli distribution**: one binary trial.
2. **Binomial distribution**: number of successes in $N$ binary trials.
3. **Multinomial distribution**: counts across $K$ mutually exclusive categories.

The goal is not just to memorize formulas. The goal is to build the mental model:

> A probability distribution is a compact model of repeated uncertainty. Maximum likelihood asks: “Which parameter values make the observed data least surprising?”

## 0. Setup

Run this once. We only need `numpy`, `math`, and `matplotlib`.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

## 1. Big picture: from one trial to many categories

Section 3.1 is about discrete random variables.

The progression is:

| Distribution | Random quantity | Example | Parameter |
|---|---:|---|---|
| Bernoulli | $x \in \{0,1\}$ | one coin flip | $\mu = P(x=1)$ |
| Binomial | $m \in \{0,\dots,N\}$ | number of heads in $N$ flips | $N, \mu$ |
| Categorical / 1-of-$K$ | $\mathbf{x}$ with one 1 and rest 0 | one class label | $\boldsymbol{\mu}$ |
| Multinomial | $m_1,\dots,m_K$ | counts for $K$ classes after $N$ draws | $N, \boldsymbol{\mu}$ |

Thinking model:

> Bernoulli is one yes/no event. Binomial compresses many yes/no events into a count. Multinomial is the same compression idea, but for more than two possible outcomes.

## 2. Bernoulli distribution

A Bernoulli random variable has two possible values:

$$
x \in \{0, 1\}
$$

Let

$$
P(x=1 \mid \mu) = \mu, \qquad P(x=0 \mid \mu) = 1 - \mu.
$$

A compact way to write both cases is:

$$
\operatorname{Bern}(x \mid \mu) = \mu^x (1-\mu)^{1-x}.
$$

Why this works:

- If $x=1$, then $\mu^1(1-\mu)^0 = \mu$.
- If $x=0$, then $\mu^0(1-\mu)^1 = 1-\mu$.

Thinking model:

> The exponent acts like a switch. The observed value selects the probability term that applies.

In [ ]:
def bernoulli_pmf(x, mu):
    x = np.asarray(x)
    return (mu ** x) * ((1 - mu) ** (1 - x))

for mu in [0.2, 0.5, 0.8]:
    xs = np.array([0, 1])
    ps = bernoulli_pmf(xs, mu)
    print(f"mu={mu}: P(x=0)={ps[0]:.2f}, P(x=1)={ps[1]:.2f}, sum={ps.sum():.2f}")

### Visualize Bernoulli probabilities

Changing $\mu$ shifts mass between $x=0$ and $x=1$.

In [ ]:
for mu in [0.2, 0.5, 0.8]:
    xs = np.array([0, 1])
    ps = bernoulli_pmf(xs, mu)
    plt.figure()
    plt.bar(xs, ps)
    plt.xticks([0, 1])
    plt.ylim(0, 1)
    plt.xlabel("x")
    plt.ylabel("probability")
    plt.title(f"Bernoulli distribution, mu={mu}")
    plt.show()

## 3. Mean and variance of Bernoulli

For Bernoulli:

$$
E[x] = \mu
$$

because $x$ is 1 with probability $\mu$ and 0 otherwise:

$$
E[x] = 1 \cdot \mu + 0 \cdot (1-\mu) = \mu.
$$

The variance is:

$$
\operatorname{var}[x] = \mu(1-\mu).
$$

Mental model for variance:

> Bernoulli variance is largest when the outcome is most uncertain. That happens at $\mu=0.5$. If $\mu$ is near 0 or 1, the outcome is almost deterministic, so variance is small.

In [ ]:
mus = np.linspace(0, 1, 201)
variances = mus * (1 - mus)

plt.figure()
plt.plot(mus, variances)
plt.xlabel("mu")
plt.ylabel("variance")
plt.title("Bernoulli variance: mu(1 - mu)")
plt.grid(True)
plt.show()

print("Maximum variance occurs near mu =", mus[np.argmax(variances)])
print("Maximum variance =", variances.max())

## 4. Maximum likelihood for Bernoulli

Suppose we observe a dataset:

$$
D = \{x_1, \dots, x_N\}, \qquad x_n \in \{0,1\}.
$$

Assuming observations are independent, the likelihood is:

$$
p(D \mid \mu) = \prod_{n=1}^N \mu^{x_n}(1-\mu)^{1-x_n}.
$$

The log likelihood is:

$$
\log p(D \mid \mu)
= \sum_{n=1}^N \left[x_n \log \mu + (1-x_n)\log(1-\mu)\right].
$$

The maximum likelihood estimate is:

$$
\mu_{ML} = \frac{1}{N}\sum_{n=1}^N x_n = \frac{m}{N}
$$

where $m$ is the number of ones.

Thinking model:

> MLE for Bernoulli says: use the observed frequency as the probability estimate.

In [ ]:
# Simulate N coin flips with true probability mu_true
mu_true = 0.7
N = 20
x = rng.binomial(n=1, p=mu_true, size=N)
mu_ml = x.mean()

print("observations:", x)
print("number of ones m:", x.sum())
print("N:", N)
print("mu_ML = m/N:", mu_ml)

### Likelihood as a function of $\mu$

The likelihood is not a probability distribution over $\mu$ here. It is a function of $\mu$ that scores how compatible each candidate $\mu$ is with the observed data.

For numerical stability, we usually work with log likelihood.

In [ ]:
def bernoulli_log_likelihood(data, mu):
    # Avoid log(0) at exactly 0 or 1.
    eps = 1e-12
    mu = np.clip(mu, eps, 1 - eps)
    return np.sum(data * np.log(mu) + (1 - data) * np.log(1 - mu))

candidate_mus = np.linspace(0.001, 0.999, 500)
ll = np.array([bernoulli_log_likelihood(x, mu) for mu in candidate_mus])

plt.figure()
plt.plot(candidate_mus, ll)
plt.axvline(mu_ml, linestyle="--")
plt.xlabel("candidate mu")
plt.ylabel("log likelihood")
plt.title("Bernoulli log likelihood")
plt.grid(True)
plt.show()

print("Grid maximum near:", candidate_mus[np.argmax(ll)])
print("Closed-form mu_ML:", mu_ml)

## 5. Sufficient statistic: why only the count matters

The Bernoulli likelihood can be rewritten using

$$
m = \sum_{n=1}^N x_n.
$$

Then:

$$
p(D \mid \mu) = \mu^m(1-\mu)^{N-m}.
$$

So the likelihood depends on the entire dataset only through $m$.

That makes $m$ a **sufficient statistic** for $\mu$.

Thinking model:

> For Bernoulli MLE, the order of flips contains no information about $\mu$. Only the total number of successes matters.

In [ ]:
data_a = np.array([1, 1, 1, 0, 0, 0, 1, 0])
data_b = np.array([0, 1, 0, 1, 1, 0, 0, 1])

print("data_a sum:", data_a.sum())
print("data_b sum:", data_b.sum())

for mu in [0.2, 0.5, 0.8]:
    lla = bernoulli_log_likelihood(data_a, mu)
    llb = bernoulli_log_likelihood(data_b, mu)
    print(f"mu={mu}: log likelihood A={lla:.6f}, B={llb:.6f}")

## 6. Small-data warning: MLE can be overconfident

If $N$ is tiny, $m/N$ can be extreme.

Example: with one observation, if $x_1=1$, then $\mu_{ML}=1$. That says future observations are certainly 1, which is often too confident.

This is not a contradiction. MLE is doing exactly what it is asked to do: maximize the likelihood of observed data. It is not automatically conservative.

This connects to later ideas:

- regularization,
- Bayesian priors,
- smoothing,
- overfitting.

In [ ]:
mu_true = 0.7
num_experiments = 10_000

for N in [1, 2, 5, 20, 100]:
    samples = rng.binomial(n=1, p=mu_true, size=(num_experiments, N))
    estimates = samples.mean(axis=1)
    print(f"N={N:3d}: mean(mu_ML)={estimates.mean():.3f}, std(mu_ML)={estimates.std():.3f}, "
          f"P(mu_ML is 0 or 1)={np.mean((estimates == 0) | (estimates == 1)):.3f}")

## 7. Binomial distribution

The Bernoulli distribution models one binary draw.

The **binomial distribution** models the count of successes in $N$ independent Bernoulli draws.

Let

$$
m = x_1 + \cdots + x_N.
$$

Then:

$$
\operatorname{Bin}(m \mid N, \mu)
= {N \choose m}\mu^m(1-\mu)^{N-m}.
$$

The combinatorial term is:

$$
{N \choose m} = \frac{N!}{(N-m)!m!}.
$$

Thinking model:

> Bernoulli likelihood for a particular sequence counts one exact pattern. Binomial probability counts all sequences with the same number of successes.

In [ ]:
def binomial_pmf(m, N, mu):
    return math.comb(N, int(m)) * (mu ** m) * ((1 - mu) ** (N - m))

N = 10
mu = 0.25
ms = np.arange(N + 1)
ps = np.array([binomial_pmf(m, N, mu) for m in ms])

print("sum of probabilities:", ps.sum())
print("most likely count:", ms[np.argmax(ps)])

plt.figure()
plt.bar(ms, ps)
plt.xlabel("m = number of successes")
plt.ylabel("probability")
plt.title("Binomial distribution: N=10, mu=0.25")
plt.grid(True, axis="y")
plt.show()

### Why the combinatorial term matters

For $N=3$ and $m=2$, these sequences all have two successes:

$$
110, \quad 101, \quad 011.
$$

Each has probability:

$$
\mu^2(1-\mu)^1.
$$

There are ${3 \choose 2}=3$ such sequences, so:

$$
P(m=2) = 3\mu^2(1-\mu).
$$

In [ ]:
from itertools import product

N = 3
mu = 0.7
all_sequences = list(product([0, 1], repeat=N))

rows = []
for seq in all_sequences:
    m = sum(seq)
    p = (mu ** m) * ((1 - mu) ** (N - m))
    rows.append((seq, m, p))

for row in rows:
    print(row)

print("\nSum probabilities where m=2:", sum(p for seq, m, p in rows if m == 2))
print("Binomial formula for m=2:", binomial_pmf(2, N, mu))

## 8. Mean and variance of Binomial

For binomial:

$$
E[m] = N\mu
$$

and

$$
\operatorname{var}[m] = N\mu(1-\mu).
$$

Why:

$$
m = x_1 + \cdots + x_N
$$

where each $x_n$ is Bernoulli. The mean of a sum is the sum of the means. For independent variables, the variance of a sum is the sum of the variances.

Thinking model:

> Binomial count is accumulated Bernoulli noise. More trials increase the absolute variance, but the fraction $m/N$ becomes more stable.

In [ ]:
mu = 0.3
num_experiments = 20_000

for N in [10, 100, 1000]:
    counts = rng.binomial(n=N, p=mu, size=num_experiments)
    fractions = counts / N
    print(f"N={N:4d}")
    print(f"  empirical E[m]       = {counts.mean():.3f}, theory = {N * mu:.3f}")
    print(f"  empirical var[m]     = {counts.var():.3f}, theory = {N * mu * (1 - mu):.3f}")
    print(f"  empirical std[m/N]   = {fractions.std():.4f}, theory = {math.sqrt(mu * (1 - mu) / N):.4f}")

## 9. The shape of the binomial distribution

The binomial distribution changes shape depending on $N$ and $\mu$.

- If $\mu=0.5$, it is symmetric.
- If $\mu$ is near 0 or 1, it is skewed.
- As $N$ grows, the fraction $m/N$ concentrates near $\mu$.

In [ ]:
settings = [(10, 0.5), (10, 0.1), (50, 0.1), (50, 0.5), (50, 0.8)]

for N, mu in settings:
    ms = np.arange(N + 1)
    ps = np.array([binomial_pmf(m, N, mu) for m in ms])
    plt.figure()
    plt.bar(ms, ps)
    plt.axvline(N * mu, linestyle="--")
    plt.xlabel("m")
    plt.ylabel("probability")
    plt.title(f"Binomial: N={N}, mu={mu}")
    plt.grid(True, axis="y")
    plt.show()

## 10. From Bernoulli to categorical variables

Binary variables have two states. Many ML labels have more than two states:

- digit classification: 10 classes,
- language modeling: vocabulary-sized classes,
- image classification: thousands of classes.

A convenient representation is **one-hot encoding**.

For $K=6$, class 3 can be represented as:

$$
\mathbf{x} = (0,0,1,0,0,0)^T.
$$

The vector satisfies:

$$
\sum_{k=1}^K x_k = 1.
$$

Let $\mu_k$ be the probability of class $k$. Then:

$$
p(\mathbf{x}\mid \boldsymbol{\mu}) = \prod_{k=1}^K \mu_k^{x_k}.
$$

This is often called the **categorical distribution**. The book presents it as the multi-state generalization underlying the multinomial distribution.

Thinking model:

> One-hot encoding turns “which class?” into a vector of switches. Exactly one switch is on.

In [ ]:
def one_hot(index, K):
    x = np.zeros(K, dtype=int)
    x[index] = 1
    return x

K = 6
x = one_hot(2, K)  # zero-based index 2 means third class
print(x)
print("sum:", x.sum())

### Probability of a one-hot observation

The product

$$
\prod_{k=1}^K \mu_k^{x_k}
$$

selects the probability of the active class.

If

$$
\boldsymbol{\mu} = (0.1, 0.2, 0.4, 0.1, 0.1, 0.1)
$$

and

$$
\mathbf{x}=(0,0,1,0,0,0)^T,
$$

then

$$
p(\mathbf{x}\mid\boldsymbol{\mu}) = 0.4.
$$

In [ ]:
def categorical_pmf(x, mu):
    x = np.asarray(x)
    mu = np.asarray(mu)
    return np.prod(mu ** x)

mu = np.array([0.1, 0.2, 0.4, 0.1, 0.1, 0.1])
x = np.array([0, 0, 1, 0, 0, 0])

print(categorical_pmf(x, mu))

## 11. Maximum likelihood for categorical variables

Suppose we observe $N$ one-hot vectors:

$$
\mathbf{x}_1, \dots, \mathbf{x}_N.
$$

Let

$$
m_k = \sum_{n=1}^N x_{nk}
$$

be the number of times class $k$ appears.

The likelihood becomes:

$$
p(D\mid\boldsymbol{\mu}) = \prod_{k=1}^K \mu_k^{m_k}.
$$

The maximum likelihood estimate is:

$$
\mu_{k,ML} = \frac{m_k}{N}.
$$

Thinking model:

> Same rule as Bernoulli: estimated probability = observed frequency. The only difference is that we now have $K$ frequencies that must sum to 1.

In [ ]:
K = 4
mu_true = np.array([0.1, 0.2, 0.6, 0.1])
N = 50

labels = rng.choice(K, size=N, p=mu_true)
X = np.eye(K, dtype=int)[labels]
counts = X.sum(axis=0)
mu_ml = counts / N

print("labels:", labels[:20], "...")
print("counts:", counts)
print("mu_ML:", mu_ml)
print("sum(mu_ML):", mu_ml.sum())

In [ ]:
plt.figure()
plt.bar(np.arange(K), mu_true, alpha=0.5, label="true probabilities")
plt.bar(np.arange(K), mu_ml, alpha=0.5, label="MLE from sample")
plt.xticks(np.arange(K))
plt.xlabel("class k")
plt.ylabel("probability")
plt.title("Categorical MLE: observed frequencies")
plt.legend()
plt.grid(True, axis="y")
plt.show()

## 12. Multinomial distribution

The multinomial distribution models counts across $K$ categories after $N$ independent draws.

Let:

$$
\mathbf{m} = (m_1,\dots,m_K), \qquad \sum_{k=1}^K m_k = N.
$$

Then:

$$
\operatorname{Mult}(m_1,\dots,m_K \mid \boldsymbol{\mu}, N)
= {N \choose m_1\,m_2\,\dots\,m_K}\prod_{k=1}^K \mu_k^{m_k}
$$

where

$$
{N \choose m_1\,m_2\,\dots\,m_K}
= \frac{N!}{m_1!m_2!\cdots m_K!}.
$$

Thinking model:

> Categorical probability scores one exact sequence. Multinomial probability sums over all sequences with the same class counts.

In [ ]:
def multinomial_coeff(counts):
    N = int(np.sum(counts))
    denom = 1
    for c in counts:
        denom *= math.factorial(int(c))
    return math.factorial(N) // denom

def multinomial_pmf(counts, mu):
    counts = np.asarray(counts, dtype=int)
    mu = np.asarray(mu, dtype=float)
    coeff = multinomial_coeff(counts)
    return coeff * np.prod(mu ** counts)

counts = np.array([2, 1, 0])
mu = np.array([0.5, 0.3, 0.2])
print("coefficient:", multinomial_coeff(counts))
print("probability:", multinomial_pmf(counts, mu))

### Example: rolling a biased die

A die has six outcomes, so $K=6$. If we roll it $N=100$ times, the outcome is a vector of six counts.

In [ ]:
K = 6
N = 100
mu_die = np.array([0.10, 0.10, 0.15, 0.20, 0.20, 0.25])
counts = rng.multinomial(N, mu_die)
mu_ml = counts / N

print("counts:", counts)
print("MLE probabilities:", np.round(mu_ml, 3))
print("true probabilities:", mu_die)

plt.figure()
faces = np.arange(1, K + 1)
plt.bar(faces - 0.15, mu_die, width=0.3, label="true")
plt.bar(faces + 0.15, mu_ml, width=0.3, label="MLE")
plt.xlabel("die face")
plt.ylabel("probability")
plt.title("Biased die: true probabilities vs MLE")
plt.legend()
plt.grid(True, axis="y")
plt.show()

## 13. Binomial as a special case of multinomial

For $K=2$, multinomial counts are:

$$
(m_1, m_2) = (m, N-m).
$$

The multinomial formula becomes:

$$
\frac{N!}{m!(N-m)!}\mu^m(1-\mu)^{N-m},
$$

which is exactly the binomial distribution.

Thinking model:

> Binomial is not separate magic. It is the two-class multinomial count distribution.

In [ ]:
N = 10
m = 3
mu_binary = 0.25

bin_p = binomial_pmf(m, N, mu_binary)
mult_p = multinomial_pmf([m, N - m], [mu_binary, 1 - mu_binary])

print("binomial:", bin_p)
print("multinomial with K=2:", mult_p)

## 14. Practical ML connection: class probabilities

In classification models, the target label is often represented as one-hot $\mathbf{x}$ or $\mathbf{t}$.

For one example, if the model predicts class probabilities $\mathbf{y}$, the likelihood of the observed one-hot target is:

$$
p(\mathbf{t}\mid\mathbf{y}) = \prod_{k=1}^K y_k^{t_k}.
$$

The negative log likelihood is:

$$
-\log p(\mathbf{t}\mid\mathbf{y}) = -\sum_{k=1}^K t_k \log y_k.
$$

That is the **cross-entropy loss** for one-hot classification.

Thinking model:

> Cross-entropy is just the negative log probability assigned to the correct class.

In [ ]:
def cross_entropy_one_hot(t, y):
    eps = 1e-12
    y = np.clip(y, eps, 1.0)
    return -np.sum(t * np.log(y))

t = np.array([0, 0, 1, 0])  # true class is class 2 in zero-based indexing
pred_good = np.array([0.05, 0.10, 0.80, 0.05])
pred_bad = np.array([0.70, 0.10, 0.10, 0.10])

print("good prediction loss:", cross_entropy_one_hot(t, pred_good))
print("bad prediction loss:", cross_entropy_one_hot(t, pred_bad))
print("-log probability of correct class, good:", -np.log(pred_good[2]))
print("-log probability of correct class, bad:", -np.log(pred_bad[2]))

## 15. Worked example: estimating bug rate from CI failures

Suppose a CI job either passes or fails.

Let:

$$
x = 1 \quad \text{if CI fails}, \qquad x = 0 \quad \text{if CI passes}.
$$

If 8 out of 100 runs fail, the Bernoulli MLE is:

$$
\mu_{ML} = \frac{8}{100} = 0.08.
$$

But the binomial distribution lets us ask a different question:

> If the true failure probability is $\mu=0.08$, how likely is it to see at least 12 failures in the next 100 runs?

In [ ]:
N = 100
mu_hat = 8 / 100
threshold = 12

prob_at_least_12 = sum(binomial_pmf(m, N, mu_hat) for m in range(threshold, N + 1))
print(f"P(M >= {threshold} | N={N}, mu={mu_hat}) = {prob_at_least_12:.4f}")

ms = np.arange(0, 25)
ps = np.array([binomial_pmf(m, N, mu_hat) for m in ms])

plt.figure()
plt.bar(ms, ps)
plt.axvline(threshold, linestyle="--")
plt.xlabel("number of failures in 100 runs")
plt.ylabel("probability")
plt.title("CI failure count under a binomial model")
plt.grid(True, axis="y")
plt.show()

## 16. Worked example: estimating traffic mix

Suppose requests are classified into four categories:

1. read,
2. write,
3. metadata,
4. admin.

We observe counts:

$$
\mathbf{m} = (700, 200, 80, 20)
$$

with $N=1000$.

The multinomial MLE is simply:

$$
\boldsymbol{\mu}_{ML} = (0.70, 0.20, 0.08, 0.02).
$$

In [ ]:
request_types = ["read", "write", "metadata", "admin"]
counts = np.array([700, 200, 80, 20])
mu_ml = counts / counts.sum()

for name, count, prob in zip(request_types, counts, mu_ml):
    print(f"{name:8s} count={count:4d}, estimated probability={prob:.3f}")

plt.figure()
plt.bar(request_types, mu_ml)
plt.ylabel("estimated probability")
plt.title("Estimated traffic mix")
plt.grid(True, axis="y")
plt.show()

## 17. Common mistakes

### Mistake 1: Confusing Bernoulli and binomial

- Bernoulli: probability of one outcome $x$.
- Binomial: probability of a count $m$ after $N$ outcomes.

### Mistake 2: Forgetting the combinatorial factor

$\mu^m(1-\mu)^{N-m}$ is the probability of one exact sequence with $m$ successes.

${N \choose m}\mu^m(1-\mu)^{N-m}$ is the probability of any sequence with $m$ successes.

### Mistake 3: Treating likelihood as a posterior probability

Likelihood $p(D\mid\mu)$ scores parameters by how well they explain the data. It is not automatically a normalized probability distribution over $\mu$.

### Mistake 4: Ignoring constraints

For multinomial parameters:

$$
\mu_k \ge 0, \qquad \sum_k \mu_k = 1.
$$

The parameters live on a simplex, not in unconstrained Euclidean space.

## 18. Summary

Section 3.1 gives the basic discrete distributions used throughout machine learning.

Core formulas:

$$
\operatorname{Bern}(x\mid\mu)=\mu^x(1-\mu)^{1-x}
$$

$$
\mu_{ML}=\frac{1}{N}\sum_{n=1}^N x_n
$$

$$
\operatorname{Bin}(m\mid N,\mu)={N\choose m}\mu^m(1-\mu)^{N-m}
$$

$$
p(\mathbf{x}\mid\boldsymbol{\mu})=\prod_{k=1}^K \mu_k^{x_k}
$$

$$
\mu_{k,ML}=\frac{m_k}{N}
$$

$$
\operatorname{Mult}(m_1,\dots,m_K\mid\boldsymbol{\mu},N)
=\frac{N!}{m_1!\cdots m_K!}\prod_{k=1}^K\mu_k^{m_k}
$$

Main mental model:

> These distributions are bookkeeping systems for uncertainty over discrete outcomes. MLE turns observed frequencies into probability estimates.

## 19. Practice problems

### Problem 1

A biased coin is flipped 20 times and lands heads 13 times. What is $\mu_{ML}$?

### Problem 2

For $N=10$ and $\mu=0.3$, compute $P(m=4)$ under the binomial distribution.

### Problem 3

A 3-class classifier sees counts $(10, 30, 60)$ in a dataset. What is $\boldsymbol{\mu}_{ML}$?

### Problem 4

For $K=3$, $N=5$, $\boldsymbol{\mu}=(0.2,0.5,0.3)$, compute the multinomial probability of counts $(1,3,1)$.

### Problem 5

Why is the order of observations irrelevant for Bernoulli MLE but relevant if the data are not i.i.d.?

In [ ]:
# Solutions

# Problem 1
print("P1:", 13 / 20)

# Problem 2
print("P2:", binomial_pmf(4, 10, 0.3))

# Problem 3
counts = np.array([10, 30, 60])
print("P3:", counts / counts.sum())

# Problem 4
print("P4:", multinomial_pmf([1, 3, 1], [0.2, 0.5, 0.3]))

# Problem 5: see markdown answer below.

Problem 5 answer:

For i.i.d. Bernoulli data, every sequence with the same number of ones has the same likelihood, so only the count matters. If the data are not independent or not identically distributed, order can carry information. For example, failures clustered at the end of a benchmark run may suggest warming, throttling, degradation, or a changing environment. Then the count alone is not sufficient.